# **Simplified AES implementation (Decryption & Encryption)**

In [36]:
SBOX = [0xE, 0x4, 0xD, 0x1,
        0x2, 0xF, 0xB, 0x8,
        0x3, 0xA, 0x6, 0xC,
        0x5, 0x9, 0x0, 0x7]
INV_SBOX = [SBOX.index(i) for i in range(16)]


def gf4_mul(a, b):
    p = 0
    for _ in range(4):
        if b & 1:
            p ^= a
        carry = a & 0x8
        a <<= 1
        if carry:
            a ^= 0b11  # x^4 + x + 1
        b >>= 1
    return p & 0xF

def mix_columns(s):
    n0, n1, n2, n3 = s
    r0 = n0 ^ gf4_mul(n2, 4)
    r1 = n1 ^ gf4_mul(n3, 4)
    r2 = gf4_mul(n0, 4) ^ n2
    r3 = gf4_mul(n1, 4) ^ n3
    return [r0, r1, r2, r3]

def inv_mix_columns(s):
    n0, n1, n2, n3 = s
    r0 = gf4_mul(n0, 9) ^ gf4_mul(n2, 2)
    r1 = gf4_mul(n1, 9) ^ gf4_mul(n3, 2)
    r2 = gf4_mul(n0, 2) ^ gf4_mul(n2, 9)
    r3 = gf4_mul(n1, 2) ^ gf4_mul(n3, 9)
    return [r0, r1, r2, r3]


def sub_bytes(s): return [SBOX[b] for b in s]
def inv_sub_bytes(s): return [INV_SBOX[b] for b in s]
def shift_rows(s): return [s[0], s[1], s[3], s[2]]
def inv_shift_rows(s): return [s[0], s[1], s[3], s[2]]
def add_round_key(s, k): return [x ^ y for x, y in zip(s, k)]


def encrypt(p, keys):
    s = add_round_key(p, keys[0])
    s = sub_bytes(s)
    s = shift_rows(s)
    s = mix_columns(s)
    s = add_round_key(s, keys[1])
    s = sub_bytes(s)
    s = shift_rows(s)
    s = mix_columns(s)
    s = add_round_key(s, keys[2])
    s = sub_bytes(s)
    s = shift_rows(s)
    s = add_round_key(s, keys[3])
    return s

def decrypt(c, keys):
    s = add_round_key(c, keys[3])
    s = inv_shift_rows(s)
    s = inv_sub_bytes(s)
    s = add_round_key(s, keys[2])
    s = inv_mix_columns(s)
    s = inv_shift_rows(s)
    s = inv_sub_bytes(s)
    s = add_round_key(s, keys[1])
    s = inv_mix_columns(s)
    s = inv_shift_rows(s)
    s = inv_sub_bytes(s)
    s = add_round_key(s, keys[0])
    return s


In [40]:
keys = [
    [0x1, 0x2, 0x3, 0x4],
    [0x5, 0x6, 0x7, 0x8],
    [0x9, 0xA, 0xB, 0xC],
    [0xD, 0xE, 0xF, 0x0]
]
plaintext = [0x0, 0x1, 0x2, 0x3]
ciphertext = encrypt(plaintext, keys)
decrypted = decrypt(ciphertext, keys)
plaintext,decrypted

([0, 1, 2, 3], [0, 1, 2, 3])

# **Integral Attack**

In [42]:
import random
random.seed(42)

def decrypt2(c, k):
    s = add_round_key(c, k)
    s = inv_shift_rows(s)
    s = inv_sub_bytes(s)
    s = inv_mix_columns(s)
    s = inv_shift_rows(s)
    s = inv_sub_bytes(s)
    return s


def recover(ciphers):
    found = []
    for k0 in range(16):
        for k1 in range(16):
            for k2 in range(16):
                for k3 in range(16):
                    key = [k0, k1, k2, k3]
                    d = [decrypt2(c, key) for c in ciphers]
                    x = [0, 0, 0, 0]
                    for s in d:
                        for i in range(4):
                            x[i] ^= s[i]
                    if x == [0, 0, 0, 0]:
                        found.append(key)
    return found


def run_once(fixed_tail, keys):
    pts = [[i] + fixed_tail for i in range(16)]
    cts = [encrypt(p, keys) for p in pts]
    res = recover(cts)
    return len(res)


In [44]:
keys = [
    [0x1, 0x2, 0x3, 0x4],
    [0x5, 0x6, 0x7, 0x8],
    [0x9, 0xA, 0xB, 0xC],
    [0xD, 0xE, 0xF, 0x0],
    [0xF, 0xE, 0xD, 0xC]  # not used
]

results = []
for _ in range(32):
    tail = [random.randint(0, 15) for _ in range(3)]
    num = run_once(tail, keys)
    results.append((tail, num))
    print("Tail =", tail, "Survived keys:", num)

avg = sum(num for _, num in results) / len(results)
percent = (avg / 65536) * 100

print("\nAverage survived keys over 32 runs:", avg)
print("That is about", round(percent, 2), "% of total keyspace")


Tail = [3, 2, 13] Survived keys: 16
Tail = [1, 0, 2] Survived keys: 21
Tail = [6, 7, 0] Survived keys: 16
Tail = [6, 13, 7] Survived keys: 21
Tail = [14, 8, 0] Survived keys: 3
Tail = [5, 13, 10] Survived keys: 12
Tail = [8, 4, 6] Survived keys: 5
Tail = [10, 3, 2] Survived keys: 16
Tail = [12, 3, 11] Survived keys: 14
Tail = [11, 8, 1] Survived keys: 12
Tail = [14, 3, 12] Survived keys: 15
Tail = [2, 9, 11] Survived keys: 21
Tail = [6, 2, 1] Survived keys: 49
Tail = [7, 9, 2] Survived keys: 7
Tail = [7, 3, 12] Survived keys: 12
Tail = [8, 14, 11] Survived keys: 28
Tail = [5, 11, 11] Survived keys: 21
Tail = [6, 8, 2] Survived keys: 16
Tail = [5, 7, 5] Survived keys: 4
Tail = [14, 12, 8] Survived keys: 12
Tail = [7, 10, 1] Survived keys: 35
Tail = [7, 1, 10] Survived keys: 16
Tail = [12, 8, 2] Survived keys: 49
Tail = [6, 10, 6] Survived keys: 16
Tail = [15, 12, 14] Survived keys: 12
Tail = [4, 8, 4] Survived keys: 16
Tail = [7, 8, 13] Survived keys: 12
Tail = [12, 11, 7] Survived keys